# Primer trim — ERP003950

Technical primer trimming for mouse IgG heavy-chain dataset `ERP003950` (`Greiff 2014`).

Source basis:
- Additional file 1: 19 FR1 forward primers + 1 IgG reverse primer
- Additional file 2: primer architecture with `NNNN` diversity region and direct adapter addition

This notebook:
- reads trimmed FASTQ from `results/ERP003950/trimmed/fastq`
- writes `results/ERP003950/pr_trimmed/{fastq,maskprimer_logs}`
- creates a self-contained primer FASTA at `/data/user/epishkin/seq_refs/erp003950_primers.fasta`

To make R1/R2 trimming robust, the FASTA includes forward primers, reverse primer, and their reverse-complement counterparts.


In [ ]:
import os, sys, sysconfig
_CONDA_ENV = '/opt/conda/envs/bcr_env'
os.environ['PATH'] = _CONDA_ENV + '/bin:' + os.environ.get('PATH', '')
os.environ['PYTHONNOUSERSITE'] = '1'
sys.path[:] = [p for p in sys.path if '/data/user/epishkin/.local' not in p]
for _site in [_CONDA_ENV + '/lib/python3.11/site-packages', sysconfig.get_path('purelib')]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ['HOME'] = '/data/user/epishkin'
os.environ['XDG_CONFIG_HOME'] = '/data/user/epishkin/.config'
os.makedirs(os.environ['XDG_CONFIG_HOME'], exist_ok=True)


In [ ]:
from pathlib import Path
import shutil
import subprocess

THREADS = 4
MAX_ERROR = 0.2
MAX_LEN = 50
FORWARD_PRIMERS = [('IgH_UAd_fw1', 'GAKGTRMAGCTTCAGGAGTC'), ('IgH_UAd_fw2', 'GAGGTBCAGCTBCAGCAGTC'), ('IgH_UAd_fw3', 'CAGGTGCAGCTGAAGSASTC'), ('IgH_UAd_fw4', 'GAGGTCCARCTGCAACARTC'), ('IgH_UAd_fw5', 'CAGGTYCAGCTBCAGCARTC'), ('IgH_UAd_fw6', 'CAGGTYCARCTGCAGCAGTC'), ('IgH_UAd_fw7', 'CAGGTCCACGTGAAGCAGTC'), ('IgH_UAd_fw8', 'GAGGTGAASSTGGTGGAATC'), ('IgH_UAd_fw9', 'GAVGTGAWGYTGGTGGAGTC'), ('IgH_UAd_fw10', 'GAGGTGCAGSKGGTGGAGTC'), ('IgH_UAd_fw11', 'GAKGTGCAMCTGGTGGAGTC'), ('IgH_UAd_fw12', 'GAGGTGAAGCTGATGGARTC'), ('IgH_UAd_fw13', 'GAGGTGCARCTTGTTGAGTC'), ('IgH_UAd_fw14', 'GARGTRAAGCTTCTCGAGTC'), ('IgH_UAd_fw15', 'GAAGTGAARSTTGAGGAGTC'), ('IgH_UAd_fw16', 'CAGGTTACTCTRAAAGWGTSTG'), ('IgH_UAd_fw17', 'CAGGTCCAACTVCAGCARCC'), ('IgH_UAd_fw18', 'GATGTGAACTTGGAAGTGTC'), ('IgH_UAd_fw19', 'GAGGTGAAGGTCATCGAGTC')]
REVERSE_PRIMER = ('IgGall_IdxX_Rv', 'CARKGGATRRRCHGATGGGG')

COMP = {'A':'T','C':'G','G':'C','T':'A','R':'Y','Y':'R','S':'S','W':'W','K':'M','M':'K','B':'V','D':'H','H':'D','V':'B','N':'N'}

def revcomp(seq):
    return ''.join(COMP[b] for b in seq[::-1])

def write_primer_fasta(vol):
    seq_dir = Path(vol) / 'seq_refs'
    seq_dir.mkdir(parents=True, exist_ok=True)
    fasta = seq_dir / 'erp003950_primers.fasta'
    entries = []
    for name, seq in FORWARD_PRIMERS:
        entries.append((name, seq))
        entries.append((name + '_rc', revcomp(seq)))
    entries.append(REVERSE_PRIMER)
    entries.append((REVERSE_PRIMER[0] + '_rc', revcomp(REVERSE_PRIMER[1])))
    with open(fasta, 'w') as fh:
        for name, seq in entries:
            fh.write(f'>{name}\n{seq}\n')
    return fasta

def run_maskprimers(cmd, logfile):
    with open(logfile, 'a') as lf:
        res = subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT)
    if res.returncode != 0:
        raise RuntimeError(f'MaskPrimers failed: {cmd}')

def run_primer_trim(volume, dataset, force=False):
    if dataset != 'ERP003950':
        raise ValueError(f'This notebook is configured only for ERP003950, got {dataset}')

    vol = Path(volume)
    primers = write_primer_fasta(vol)
    src_dir = vol / 'results' / dataset / 'trimmed' / 'fastq'
    if not src_dir.is_dir():
        raise FileNotFoundError(f'Trimmed FASTQ dir not found: {src_dir}')

    base = vol / 'results' / dataset / 'pr_trimmed'
    out_dir = base / 'fastq'
    logs_dir = base / 'maskprimer_logs'

    if force and base.exists():
        shutil.rmtree(base)

    out_dir.mkdir(parents=True, exist_ok=True)
    logs_dir.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(
        f.name.replace('_1.trim.fastq.gz', '').replace('_2.trim.fastq.gz', '')
        for f in src_dir.glob('*.trim.fastq.gz')
    ))
    print(f'[primer_trim] {dataset}: {len(pairs)} pairs; primers={primers}')

    for base_name in pairs:
        for mate in ('1', '2'):
            src = src_dir / f'{base_name}_{mate}.trim.fastq.gz'
            if not src.exists():
                print(f'  [{base_name}] skip mate {mate}: missing input')
                continue
            expected = out_dir / f'{base_name}_{mate}.pr.fastq.gz'
            if expected.exists():
                print(f'  [{base_name}_{mate}] already done, skip')
                continue
            outname = f'{base_name}_{mate}.pr'
            cmd = [
                'MaskPrimers.py', 'align',
                '-s', str(src),
                '-p', str(primers),
                '--mode', 'cut',
                '--maxerror', str(MAX_ERROR),
                '--nproc', str(THREADS),
                '--maxlen', str(MAX_LEN),
                '--outdir', str(out_dir),
                '--outname', outname,
            ]
            print(f'  [{base_name}_{mate}] MaskPrimers ...')
            run_maskprimers(cmd, logs_dir / f'{base_name}_{mate}.maskprimers.log')

    for f in sorted(out_dir.glob('*_primers-pass.fastq.gz')):
        target = out_dir / f.name.replace('_primers-pass', '')
        f.rename(target)

    n = len(list(out_dir.glob('*.pr.fastq.gz')))
    print(f'[primer_trim] DONE: {n} pr-trimmed files')


### Run

Run after `adapter_trim_mouse.ipynb` has produced `*.trim.fastq.gz` files.


In [ ]:
run_primer_trim('/data/user/epishkin', 'ERP003950', force=False)
